# Saving Yolked Experiment Data
Run this file to save the results of a yolked experiment, which had a code segment pasted in from a selection experiment.


In [41]:
import pandas as pd
# import numpy as np
from typing import Union
from collections.abc import Iterable, Generator
from pathlib import Path
from enum import Enum
import json
import itertools
from dataclasses import dataclass, field, asdict
import os

## The Selection Experiment Output File
This is the only line that you will need to edit.

In [42]:
raw_results_path: Path = Path('./raw_yol_data.csv')
selection_results_path: Path = Path('./results/saved_yolked_data.csv')

## Extract data from path

In [43]:
class task_type(Enum):
    yoked_trial = 'yoked_trial'
    categorize_trial = 'categorize_trial'
    fixation = 'fixation'
    test_trial = 'test_trial'
    test_rating = 'test_rating'
    welcome = 'welcome'
    unknown = 'unknown'
    "This task type is for all that could not be categorized successfully as a task_type"

def interpret_name_as_task_type(task:str)->task_type:
    match str(task).lower():
        case 'yoked_trial': return task_type.yoked_trial
        case 'categorize_trial': return task_type.categorize_trial
        case 'fixation': return task_type.fixation
        case 'test_trial': return task_type.test_trial
        case 'test_rating': return task_type.test_rating
        case 'welcome': return task_type.welcome
        case _: return task_type.unknown

@dataclass(frozen=True,slots=True)
class experiment_metadata:
    UUID: str
    "A UUID to identify the experiment"
    category_boundary: str
    "The boundary between the two categories in the experiment (save vs. not)"
    learning_stimuli: list[list[str]]
    "All stimuli selected from learning. This is a float as a string."
    testing_stimuli: list[list[str]]
    "All stimuli used for testing. This is a float as a string. First axis should be the same size as learning_stimuli"
    testing_individual_results: list[list[bool]]
    "The categorizations made by the participant for each of the corresponting stimuli. <br>True='safe'=`> category boundary`  <br>False='not safe' = `< category boundary`"
    testing_individual_correctness: list[list[bool]]
    "Whether the individual correctly classified the testing stimuli. If True they categorized correctly."
    testing_round_correctness: list[float]
    "Combines correctness by round, giving the percentage correct"
    testing_round_certainties: list[int]
    "Reports the participant certainty ratings (0-4) by round"
    testing_overall_correctness: float
    "Combines correctness for the entirety of all tests, giving overall percentage correct"

    # @property
    # def yoked_jspsych_config(self)->str:
    #     "This generates a large string to be pasted into the yoked jspsych document to configure it."
    #     s = []
    #     s.append("")
    #     s.append(f"const CATEGORY_BOUNDARY = {self.category_boundary};")
    #     s.append(f'const SELECTION_UUID = "{self.UUID}";')
    #     s.append("")
    #     s.append("const YOKED_STIMULI = [")
    #     for ls_round in self.learning_stimuli:
    #         row = ["\t["]
    #         for ls in ls_round:
    #             row.append(f"{ls}")
    #             row.append(", ")
    #         row.pop()
    #         row.append("],")
    #         s.append("".join(row))
    #     s.append("\t];")
    #     s.append("")
    #     s.append("const TEST_STIMULI = [")
    #     for ts_round in self.testing_stimuli:
    #         row = ["\t["]
    #         for ts in ts_round:
    #             row.append(f"{ts}")
    #             row.append(", ")
    #         row.pop()
    #         row.append("],")
    #         s.append("".join(row))
    #     s.append("\t];")
    #     s.append("")
    #
    #     return "\n".join(s)

In [44]:
working_results: pd.DataFrame = pd.read_csv(raw_results_path)
# Only keep the columns I care about
useful_columns = ['task','response','correct_answer','correct','roundness','UUID','category_boundary']
working_results = working_results[useful_columns] #keep only useful columns
# print(working_results)

Now, we will define a bunch of dataclasses to describe the relevant data they contain.

In [45]:
@dataclass(frozen=True,slots=False)
class task_yoked_trial:
    selected_roundness:str

@dataclass(frozen=True,slots=False)
class task_categorize_trial:
    pass

@dataclass(frozen=True,slots=False)
class task_fixation:
    pass

@dataclass(frozen=True,slots=False)
class task_test_trial:
    roundness:str
    correct_response:str
    chosen_response:str
    chosen_category:bool
    "This is True if greater than boundary, i.e. 'safe'"
    was_correct:bool

@dataclass(frozen=True,slots=False)
class task_test_rating:
    confidence_rating:int

@dataclass(frozen=True,slots=False)
class task_welcome:
    UUID:str
    category_boundary:str

@dataclass(frozen=True,slots=False)
class task_unknown:
    task_index:int


type all_tasks = Union[
    task_yoked_trial,
    task_categorize_trial,
    task_fixation,
    task_test_trial,
    task_test_rating,
    task_welcome,
    task_unknown
    ]

Now, we will structure that data from the pandas dataframe

In [46]:
# useful_columns = ['task','response','correct_answer','correct','roundness','UUID','category_boundary']

def extract_typed_tasks(df: pd.DataFrame) -> Generator[all_tasks,None,None]:
    for i, (_, row) in enumerate(df.iterrows()):
        row_task = interpret_name_as_task_type(row['task'])
        task:all_tasks
        match row_task:
            case task_type.yoked_trial:
                task = task_yoked_trial(selected_roundness=row['roundness'])
            case task_type.categorize_trial:
                task = task_categorize_trial()
            case task_type.fixation:
                task = task_fixation()
            case task_type.test_trial:
                task = task_test_trial(
                    roundness=row['roundness'],
                    correct_response=row['correct_answer'],
                    chosen_response=row['response'],
                    chosen_category= True if row['response'] == 'f' else False, # True = Safe
                    was_correct=bool(row['correct']),
                    )
            case task_type.test_rating:
                response = row['response'] # Looks like `"{""Q0"":2}"` (double-quotes automatically escaped by pandas)
                confidence:int = json.loads(response)["Q0"]
                task = task_test_rating(confidence_rating=confidence)
            case task_type.welcome:
                task = task_welcome(
                    UUID= row['UUID'],
                    category_boundary= row['category_boundary']
                )
            case task_type.unknown | _:
                task = task_unknown(task_index=int(i))
        yield task

def split_tasks_to_experiment_rounds(tasks:Iterable[all_tasks])->Generator[list[all_tasks],None,None]:
    """
    This assumes each round is started with (1) selection_trials, then (2) test_trials, then finally (3) one test_rating. 
    Anything not following this structure will be ignored. Only these objects are returned in the order they occour.
    Any additional elements not ending in a test_rating will be ignored.
    """
    experiment_round = []
    
    for t in tasks:
        if not isinstance(t,(task_yoked_trial,task_test_trial,task_test_rating)):
            continue
        experiment_round.append(t)
        if isinstance(t,task_test_rating):
            yield experiment_round
            experiment_round = []

@dataclass
class exp_params:
    UUID:str
    category_boundary:str

def extract_experiment_global_parameters(tasks:Iterable[all_tasks])->exp_params:
    # Pull this information out of tasks.
    for t in tasks:
        if isinstance(t, task_welcome):
            return exp_params(
                UUID=t.UUID,
                category_boundary=t.category_boundary
            )

    raise ValueError("There was no 'welcome' task to provide needed parameters in the entire experiment.")

def extract_experiment_metadata(experiment: pd.DataFrame)->experiment_metadata:
    tasks = extract_typed_tasks(experiment)
    t1, t2 = itertools.tee(tasks,2)
    global_parameters = extract_experiment_global_parameters(tasks=t1)
    experiment_rounds = split_tasks_to_experiment_rounds(tasks=t2)

    learning_stimuli:list[list[str]]                = []
    testing_stimuli:list[list[str]]                 = []
    testing_individual_results:list[list[bool]]     = []
    testing_individual_correctness:list[list[bool]] = []
    testing_round_correctness:list[float]           = []
    testing_round_certainties:list[int]             = []
    testing_overall_correctness:float               = 0

    for round in experiment_rounds:
        l_stim = []
        t_stim = []
        t_results = []
        t_correct = []
        for trial in round:
            if isinstance(trial,task_yoked_trial):
                l_stim.append(trial.selected_roundness)
                continue
            if isinstance(trial,task_test_trial):
                t_stim.append(trial.roundness)
                t_results.append(trial.chosen_category)
                t_correct.append(trial.was_correct)

            if isinstance(trial,task_test_rating):
                testing_round_certainties.append(trial.confidence_rating)
                continue

        learning_stimuli.append(l_stim)
        testing_stimuli.append(t_stim)
        testing_individual_results.append(t_results)
        testing_individual_correctness.append(t_correct)
        testing_round_correctness.append(sum(t_correct)/len(t_correct))

        all_correctness = list(itertools.chain.from_iterable(testing_individual_correctness))
        testing_overall_correctness = sum(all_correctness)/len(all_correctness)

    return experiment_metadata(
        UUID=global_parameters.UUID,
        category_boundary=global_parameters.category_boundary,
        learning_stimuli=learning_stimuli,
        testing_stimuli=testing_stimuli,
        testing_individual_results=testing_individual_results,
        testing_individual_correctness=testing_individual_correctness,
        testing_round_correctness=testing_round_correctness,
        testing_round_certainties=testing_round_certainties,
        testing_overall_correctness=testing_overall_correctness
    )

Finally, we extract all useful information from our experimental data.

In [47]:
experiment_data = extract_experiment_metadata(working_results)

## Save Results
Now that we save these results to the end of the selection CSV file.

In [48]:

dict_result = asdict(experiment_data)
# dict_result['yoked_clipboard_config'] = experiment_data.yoked_jspsych_config
# dict_result['as_json'] = json.dumps(dict_result)

def save_row(result, filename):
    write_header = not os.path.exists(filename)
    pd.DataFrame([result]).to_csv(filename, mode="a", header=write_header, index=False)

save_row(dict_result,selection_results_path)
# print("/**************** START PASTE ******************/")
# print(experiment_data.yoked_jspsych_config)
# print("/***************** END PASTE *******************/")